# ScholarAI v3 SOTA: Llama-3 8B Backend
This notebook deploys the **Deep Semantic Evasion** engine. It uses **AMR Graph Parsing** to strip AI syntax and **Llama-3 8B** (pivoted from Gemma-4 for stability) to generate human-like academic text.

### ⚠️ Prerequisites
Before running, click the **Key icon (Secrets)** on the left and add:
1. `HF_TOKEN`: Your HuggingFace token (must have access to Llama-3).
2. `NGROK_TOKEN`: Your ngrok authentication token.
3. **Enable "Notebook access"** for both.

In [ ]:
!pip install --upgrade amrlib fastapi uvicorn "pydantic<=2.12.3" accelerate bitsandbytes \
    python-multipart pyngrok penman unidecode huggingface_hub sentencepiece protobuf \
    word2number celery redis python-jose[cryptography] passlib[bcrypt] slowapi \
    "numpy<2.1" "pillow<12.0"
!pip install git+https://github.com/huggingface/transformers.git

import os
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok
from huggingface_hub import login

# --- STEP 1: ENVIRONMENT SETUP ---
print("🚀 Initializing Kaggle Environment...")
%cd /kaggle/working
!rm -rf sensorspine-humaniser-v3
!apt-get update && apt-get install -y redis-server > /dev/null




# --- STEP 2: AUTH & CLONE ---
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
NGROK_TOKEN = user_secrets.get_secret("NGROK_TOKEN")

login(token=HF_TOKEN)

!git clone https://github.com/NandishSinha1403/sensorspine-humaniser-v3.git
%cd sensorspine-humaniser-v3/backend


print("\n📥 Downloading AMR Semantic Models...")
!mkdir -p models
!wget -q --show-progress https://github.com/bjascob/amrlib-models/releases/download/parse_xfm_bart_base-v0_1_0/model_parse_xfm_bart_base-v0_1_0.tar.gz
!tar -xzf model_parse_xfm_bart_base-v0_1_0.tar.gz -C models/ && mv models/model_parse_xfm_bart_base-v0_1_0 models/model_stog
!wget -q --show-progress https://github.com/bjascob/amrlib-models/releases/download/model_generate_t5wtense-v0_1_0/model_generate_t5wtense-v0_1_0.tar.gz
!tar -xzf model_generate_t5wtense-v0_1_0.tar.gz -C models/ && mv models/model_generate_t5wtense-v0_1_0 models/model_gtos

ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
public_url = ngrok.connect(8000).public_url

print(f"\n================================================")
print(f"🚀 KAGGLE BACKEND URL: {public_url}")
print(f"================================================\n")

!chmod +x start.sh
!./start.sh


  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.3/353.3 kB 9.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.9/70.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 30.0 MB/s eta 0:00:00
   ━━━━

### Phase 3: Causal Language Modeling (CLM) Fine-Tuning
This will adapt Qwen2-7B to the pre-AI academic style using the 790 preprocessed papers.

In [ ]:
!python3 fine_tune_clm.py

### Phase 4: Direct Preference Optimization (DPO)
This will penalize the model for generating AI-predictable structures using the DiagnosticJudge.

In [ ]:
!python3 fine_tune_dpo.py